# Phase 5 — Final LSTM RUL Training

**Starting point:** Phase 4 severity checkpoint (`phase4_severity.keras`)  
**RUL architecture:** independent LSTM branch processing the same 128×24 raw window  
**Detection / Diagnosis / Severity:** copied from Phase 4 and frozen  
**RUL:** trainable LSTM + RUL head + 2-feature RUL auxiliary input

This notebook is the **final single-run version** of the Phase 5 reference notebook.
The Hyperband / HP-search section is intentionally removed. This version uses a stronger 64-unit, 2-layer LSTM and a Huber+MAE RUL objective to give the final run more capacity and directly optimize the validation metric.

**Phase 4 is not modified by this notebook.**

In [1]:
import sys, os, json, warnings, time
warnings.filterwarnings("ignore")

sys.path.insert(0, "/Users/harsh/Documents/UAV_Engine/validation")

import numpy as np
import tensorflow as tf
from tensorflow import keras

from tf_data_pipeline import make_dataset
from model_architectures import (
    build_encoder_input,
    build_encoder,
    build_detection_head,
    build_diagnosis_head,
    build_severity_head,
    build_rul_head,
    build_rul_lstm_encoder,
)

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.16.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
DATA_DIR = "/Users/harsh/Documents/UAV_Engine/validation/chunks"
INDEX_PATH = os.path.join(DATA_DIR, "scenario_index.json")
SCALER_PATH = "/Users/harsh/Documents/UAV_Engine/validation/models/scaler.pkl"

# Phase 4 is READ-ONLY.
PHASE4_CKPT = "/Users/harsh/Documents/UAV_Engine/validation/models/phase4_severity.keras"

# IMPORTANT:
# Continue training ON TOP OF the model that produced the user's
# current best validation RUL MAE = 0.74562 hours (Epoch 2).
START_CKPT = (
    "/Users/harsh/Documents/UAV_Engine/validation/models/"
    "phase5_rul_manual_hp_temp.keras"
)

# Candidate checkpoint for this continuation run.
CANDIDATE_CKPT = (
    "/Users/harsh/Documents/UAV_Engine/validation/models/"
    "phase5_rul_manual_hp_continue_temp.keras"
)

# Only promote to this file if the candidate beats the old best.
FINAL_CKPT = (
    "/Users/harsh/Documents/UAV_Engine/validation/models/"
    "phase5_rul_manual_hp.keras"
)

BEST_MAE_RECORD = (
    "/Users/harsh/Documents/UAV_Engine/validation/models/"
    "phase5_rul_manual_hp_best_mae.json"
)

CLASS_WEIGHTS_PATH = (
    "/Users/harsh/Documents/UAV_Engine/validation/models/"
    "diag_class_weights_keras.npy"
)

# The checkpoint being continued is the 32-unit / 2-layer LSTM model
# shown in the user's training output.
LSTM_UNITS = 32
LSTM_LAYERS = 2
DROPOUT = 0.10

# Lower LR because Epoch 2 already reached 0.74562 and later epochs worsened.
LEARNING_RATE = 5e-5

BATCH_SIZE = 128
STEPS_PER_EPOCH = 4196
VALIDATION_STEPS = 522
TEST_STEPS = 522
EPOCHS = 20

# This is the known best from the current run.
KNOWN_BEST_VAL_MAE = 0.74562

print("START CHECKPOINT:", START_CKPT)
print("FINAL CHECKPOINT:", FINAL_CKPT)
print()
print("Continuation parameters:")
print("  LSTM units       =", LSTM_UNITS)
print("  LSTM layers      =", LSTM_LAYERS)
print("  dropout          =", DROPOUT)
print("  learning rate    =", LEARNING_RATE)
print("  batch size       =", BATCH_SIZE)
print("  continuation epochs =", EPOCHS)
print("  previous best val MAE =", KNOWN_BEST_VAL_MAE)


START CHECKPOINT: /Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp_temp.keras
FINAL CHECKPOINT: /Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp.keras

Continuation parameters:
  LSTM units       = 32
  LSTM layers      = 2
  dropout          = 0.1
  learning rate    = 5e-05
  batch size       = 128
  continuation epochs = 20
  previous best val MAE = 0.74562


## 1. Losses and metrics

These are kept consistent with the Phase 4 / Phase 5 reference notebooks.
RUL samples are masked using `rul_valid`.

In [3]:
class_weights_np = np.load(CLASS_WEIGHTS_PATH)
class_weights_tf = tf.constant(class_weights_np, dtype=tf.float32)


def weighted_diagnosis_loss(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    bsz = tf.shape(y_true_int)[0]

    channel_idx = tf.tile(
        tf.range(8)[tf.newaxis, :],
        [bsz, 1]
    )

    gather_idx = tf.stack(
        [channel_idx, y_true_int],
        axis=-1
    )

    sw = tf.gather_nd(class_weights_tf, gather_idx)

    y_true_onehot = tf.one_hot(
        y_true_int,
        depth=6
    )

    ce = -tf.reduce_sum(
        y_true_onehot * tf.math.log(y_pred + 1e-7),
        axis=-1
    )

    return tf.reduce_mean(ce * sw)


def diagnosis_accuracy(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    y_pred_class = tf.cast(
        tf.argmax(y_pred, axis=-1),
        tf.int32
    )

    return tf.reduce_mean(
        tf.cast(
            tf.equal(y_true_int, y_pred_class),
            tf.float32
        )
    )


def nonzero_severity_mse(y_true, y_pred):
    mask = tf.cast(y_true > 0, tf.float32)

    sq_err = tf.square(
        y_pred - y_true
    ) * mask

    return tf.reduce_sum(sq_err) / (
        tf.reduce_sum(mask) + 1e-7
    )

print("Losses and metrics defined.")

Losses and metrics defined.


2026-08-30 08:17:38.477675: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-08-30 08:17:38.478014: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-08-30 08:17:38.480244: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-08-30 08:17:38.480910: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-30 08:17:38.480987: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## 2. Build datasets

This is the same masked multi-output contract used by the Phase 5 reference.

`x_rul_aux` has **2 features**, matching the reference Phase 5 model:
the auxiliary RUL information supplied by `tf_data_pipeline`.

In [4]:
def to_multi_output_with_mask(inputs, labels):
    x_dict = {
        "x": inputs["x"],
        "x_rul_aux": inputs["x_rul_aux"],
    }

    y_dict = {
        "y_detection": labels["y_detection"],
        "y_diagnosis": labels["y_diagnosis"],
        "y_severity": labels["y_severity"],
        "y_rul_hours": labels["y_rul_hours"],
    }

    ones = tf.ones_like(labels["y_detection"])

    sw_dict = {
        "y_detection": ones,
        "y_diagnosis": ones,
        "y_severity": ones,
        "y_rul_hours": tf.cast(
            labels["rul_valid"],
            tf.float32
        ),
    }

    return x_dict, y_dict, sw_dict


# Fresh datasets for this final run.
train_ds = make_dataset(
    INDEX_PATH,
    DATA_DIR,
    "train",
    SCALER_PATH,
    batch_size=BATCH_SIZE,
).map(to_multi_output_with_mask)

val_ds = make_dataset(
    INDEX_PATH,
    DATA_DIR,
    "val",
    SCALER_PATH,
    batch_size=BATCH_SIZE,
    shuffle_buffer=0,
).map(to_multi_output_with_mask)

print("Training and validation datasets built.")

Training and validation datasets built.


## 3. Build the Phase 4 + independent LSTM model

The architecture follows the Phase 5 reference notebook exactly:

- Phase 4 TCN → Detection
- Phase 4 TCN → Diagnosis
- Phase 4 TCN → Severity
- **Independent LSTM** → RUL
- `x_rul_aux` → RUL head

The Phase 4 checkpoint is loaded **only for matching weights**.
No Phase 4 file is modified.

In [5]:
if not os.path.exists(START_CKPT):
    raise FileNotFoundError(
        f"Starting checkpoint not found: {START_CKPT}\n"
        "Stop the current training first so its best temp checkpoint is available."
    )

print("Loading current best model from:")
print(START_CKPT)

# Load weights/model structure from the current best checkpoint.
# compile=False avoids restoring the old optimizer state; we intentionally
# restart with a smaller LR for fine-tuning from the best Epoch-2 weights.
model = keras.models.load_model(
    START_CKPT,
    safe_mode=False,
    compile=False,
)

# Protect Phase 4. Only the independent RUL branch remains trainable.
for layer in model.layers:
    if layer.name.startswith("rul_"):
        layer.trainable = True
    else:
        layer.trainable = False

print("Loaded model:", model.name)
print("Trainable variables:", len(model.trainable_variables))
print("Expected: only independent RUL LSTM + RUL head variables")
model.summary()


Loading current best model from:
/Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp_temp.keras
Loaded model: functional
Trainable variables: 12
Expected: only independent RUL LSTM + RUL head variables


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 24)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv1 (Conv1D) │ (None, 128, 32)   │      2,336 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop1          │ (None, 128, 32)   │          0 │ tcn0_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv2 (Conv1D) │ (None, 128, 32)   │      3,104 │ tcn0_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop2          │ (None, 128, 32)   │          0 │ tcn0_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_proj (Conv1D)  │ (None, 128, 32)   │        800 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_add (Add)      │ (None, 128, 32)   │          0 │ tcn0_drop2[0][0], │
│                     │                   │            │ tcn0_proj[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_relu (ReLU)    │ (None, 128, 32)   │          0 │ tcn0_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv1 (Conv1D) │ (None, 128, 32)   │      3,104 │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop1          │ (None, 128, 32)   │          0 │ tcn1_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv2 (Conv1D) │ (None, 128, 32)   │      3,104 │ tcn1_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop2          │ (None, 128, 32)   │          0 │ tcn1_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 32)   │          0 │ tcn1_drop2[0][0], │
│                     │                   │            │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu (ReLU)    │ (None, 128, 32)   │          0 │ tcn1_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv1 (Conv1D) │ (None, 128, 32)   │      3,104 │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop1          │ (None, 128, 32)   │          0 │ tcn2_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv2 (Conv1D) │ (None, 128, 32)   │      3,104 │ tcn2_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop2          │ (None, 128, 32)   │          0 │ tcn2_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 32)   │          0 │ tcn2_drop2[0][0], │
│                     │                   │            │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu (ReLU)    │ (None, 128, 32)   │          0 │ tcn2_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 69,530 (271.60 KB)

 Trainable params: 20,225 (79.00 KB)

 Non-trainable params: 49,305 (192.60 KB)

## 4. Warm-start from Phase 4 and freeze the Phase 4 path

Only matching Phase 4 layers receive Phase 4 weights.

All layers belonging to Detection / Diagnosis / Severity remain frozen.
The independent `rul_` LSTM and RUL head remain trainable.

In [6]:
if not os.path.exists(PHASE4_CKPT):
    raise FileNotFoundError(
        f"Phase 4 checkpoint not found: {PHASE4_CKPT}"
    )

phase4_model = keras.models.load_model(
    PHASE4_CKPT,
    safe_mode=False,
    compile=False,
)

phase4_layers = {
    layer.name: layer
    for layer in phase4_model.layers
}

n_matched = 0
n_skipped = 0

for layer in model.layers:
    if layer.name in phase4_layers:
        try:
            layer.set_weights(
                phase4_layers[layer.name].get_weights()
            )
            n_matched += 1
        except ValueError as e:
            n_skipped += 1
            print(
                f"  skipped {layer.name}: {e}"
            )

print(
    f"Warm-started {n_matched} matching layers "
    f"from Phase 4."
)

if n_skipped:
    print(f"Skipped {n_skipped} incompatible matching layer(s).")

# Freeze everything except the independent RUL branch.
for layer in model.layers:
    if (
        layer.name.startswith("rul_")
        or layer.name == "x_rul_aux"
    ):
        layer.trainable = True
    else:
        layer.trainable = False

print()
print(
    "Trainable variables:",
    len(model.trainable_variables)
)
print(
    "Non-trainable variables:",
    len(model.non_trainable_variables)
)

Warm-started 52 matching layers from Phase 4.

Trainable variables: 12
Non-trainable variables: 58


## 5. Compile

Same loss weights as the Phase 5 reference notebook.

In [7]:
def rul_huber_mae_loss(y_true, y_pred):
    # Robust RUL objective for fine-tuning around the current best solution.
    err = y_true - y_pred
    abs_err = tf.abs(err)
    delta = tf.constant(0.5, dtype=tf.float32)

    huber = tf.where(
        abs_err <= delta,
        0.5 * tf.square(err),
        delta * (abs_err - 0.5 * delta),
    )

    # Equal weight on robust regression and the metric we actually care about.
    return 0.5 * tf.reduce_mean(huber) + 0.5 * tf.reduce_mean(abs_err)


model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0,
    ),
    loss={
        "y_detection": keras.losses.BinaryCrossentropy(),
        "y_diagnosis": weighted_diagnosis_loss,
        "y_severity": keras.losses.MeanSquaredError(),
        "y_rul_hours": rul_huber_mae_loss,
    },
    loss_weights={
        "y_detection": 1.0,
        "y_diagnosis": 50.0,
        "y_severity": 20.0,
        "y_rul_hours": 1.0,
    },
    metrics={
        "y_detection": keras.metrics.BinaryAccuracy(name="acc"),
        "y_diagnosis": diagnosis_accuracy,
        "y_severity": nonzero_severity_mse,
        "y_rul_hours": keras.metrics.MeanAbsoluteError(name="mae"),
    },
)

print("Model compiled for continuation fine-tuning.")


Model compiled for continuation fine-tuning.


## 6. Final training — single run

No Hyperband and no repeated HP trials.

The best checkpoint is selected by **validation RUL MAE**.

In [8]:
# Never overwrite the starting checkpoint.
if os.path.exists(CANDIDATE_CKPT):
    os.remove(CANDIDATE_CKPT)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        CANDIDATE_CKPT,
        monitor="val_y_rul_hours_mae",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_y_rul_hours_mae",
        mode="min",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_y_rul_hours_mae",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

print("\n" + "=" * 60)
print("CONTINUING FROM CURRENT BEST CHECKPOINT")
print("=" * 60)
print(f"Starting MAE: {KNOWN_BEST_VAL_MAE:.5f} hours")
print(f"Learning rate: {LEARNING_RATE}")
print("The original checkpoint will NOT be overwritten.")
print("=" * 60 + "\n", flush=True)

t0 = time.time()

history = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

elapsed = time.time() - t0
val_maes = history.history["val_y_rul_hours_mae"]
new_best_mae = float(np.min(val_maes))
best_epoch = int(np.argmin(val_maes)) + 1

print("\n" + "=" * 60)
print("CONTINUATION RESULT")
print("=" * 60)
print(f"Starting/baseline val RUL MAE : {KNOWN_BEST_VAL_MAE:.6f} hours")
print(f"New best val RUL MAE         : {new_best_mae:.6f} hours")
print(f"Best continuation epoch      : {best_epoch}")
print(f"Improvement                  : {KNOWN_BEST_VAL_MAE - new_best_mae:.6f} hours")
print(f"Training time                : {elapsed / 60:.1f} minutes")
print("=" * 60)



CONTINUING FROM CURRENT BEST CHECKPOINT
Starting MAE: 0.74562 hours
Learning rate: 5e-05
The original checkpoint will NOT be overwritten.

Epoch 1/20


2026-08-30 08:17:40.928442: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


4196/4196 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 0.4781 - y_detection_acc: 0.9789 - y_detection_loss: 0.0781 - y_diagnosis_diagnosis_accuracy: 0.9885 - y_diagnosis_loss: 4.4347e-04 - y_rul_hours_loss: 0.3472 - y_rul_hours_mae: 0.7634 - y_severity_loss: 0.0015 - y_severity_nonzero_severity_mse: 0.0108
Epoch 1: val_y_rul_hours_mae improved from None to 0.76108, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp_continue_temp.keras

Epoch 1: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp_continue_temp.keras
4196/4196 ━━━━━━━━━━━━━━━━━━━━ 512s 120ms/step - loss: 0.4781 - y_detection_acc: 0.9789 - y_detection_loss: 0.0781 - y_diagnosis_diagnosis_accuracy: 0.9885 - y_diagnosis_loss: 4.4347e-04 - y_rul_hours_loss: 0.3472 - y_rul_hours_mae: 0.7634 - y_severity_loss: 0.0015 - y_severity_nonzero_severity_mse: 0.0108 - val_loss: 0.5715 - val_y_detection_acc: 0.9843 - val_y_detection_loss: 0.0555 - val_y_d

## 7. Validation results

In [9]:
h = history.history

print("Available history keys:")
print(list(h.keys()))
print()

new_best_mae = min(
    h["val_y_rul_hours_mae"]
)

print(
    "Best val_y_detection_acc:",
    max(h["val_y_detection_acc"])
)

print(
    "Best val_y_diagnosis_diagnosis_accuracy:",
    max(
        h[
            "val_y_diagnosis_diagnosis_accuracy"
        ]
    )
)

print(
    "Best val_y_severity_nonzero_severity_mse:",
    min(
        h[
            "val_y_severity_nonzero_severity_mse"
        ]
    )
)

print(
    "Best val_y_rul_hours_mae:",
    new_best_mae
)

Available history keys:
['loss', 'val_loss', 'val_y_detection_acc', 'val_y_detection_loss', 'val_y_diagnosis_diagnosis_accuracy', 'val_y_diagnosis_loss', 'val_y_rul_hours_loss', 'val_y_rul_hours_mae', 'val_y_severity_loss', 'val_y_severity_nonzero_severity_mse', 'y_detection_acc', 'y_detection_loss', 'y_diagnosis_diagnosis_accuracy', 'y_diagnosis_loss', 'y_rul_hours_loss', 'y_rul_hours_mae', 'y_severity_loss', 'y_severity_nonzero_severity_mse', 'learning_rate']

Best val_y_detection_acc: 0.9843301177024841
Best val_y_diagnosis_diagnosis_accuracy: 0.9907993674278259
Best val_y_severity_nonzero_severity_mse: 0.004970240872353315
Best val_y_rul_hours_mae: 0.4991084933280945


## 8. Checkpoint safeguard

The existing best Phase 5 LSTM checkpoint is **never overwritten unless this run actually improves validation MAE**.

In [10]:
# ============================================================
# PROMOTE ONLY IF THE CONTINUATION BEATS THE EXISTING BEST
# ============================================================
if not os.path.exists(CANDIDATE_CKPT):
    raise FileNotFoundError(
        "No candidate checkpoint was produced."
    )

# Prefer the recorded best if it exists and is valid; otherwise use the
# known 0.74562-hour best from the user's current training output.
old_best_mae = KNOWN_BEST_VAL_MAE

if os.path.exists(BEST_MAE_RECORD):
    try:
        with open(BEST_MAE_RECORD) as f:
            recorded = json.load(f)
        recorded_mae = recorded.get(
            "best_val_rul_hours_mae",
            recorded.get("best_val_rul_mae_hours", None),
        )
        if recorded_mae is not None:
            old_best_mae = min(old_best_mae, float(recorded_mae))
    except Exception:
        pass

print("Existing best validation RUL MAE:", old_best_mae)
print("Continuation best validation RUL MAE:", new_best_mae)

if new_best_mae < old_best_mae:
    os.replace(CANDIDATE_CKPT, FINAL_CKPT)

    with open(BEST_MAE_RECORD, "w") as f:
        json.dump(
            {
                "best_val_rul_hours_mae": float(new_best_mae),
                "best_epoch_in_continuation": int(best_epoch),
                "previous_best_val_rul_hours_mae": float(old_best_mae),
                "lstm_units": LSTM_UNITS,
                "lstm_layers": LSTM_LAYERS,
                "dropout": DROPOUT,
                "learning_rate": LEARNING_RATE,
                "source_checkpoint": START_CKPT,
            },
            f,
            indent=2,
        )

    print("\n>>> NEW BEST MAE — promoted to:")
    print(FINAL_CKPT)
else:
    os.remove(CANDIDATE_CKPT)

    print("\n>>> NOT BETTER — candidate deleted.")
    print(">>> Existing best checkpoint was left untouched.")


Existing best validation RUL MAE: 0.74562
Continuation best validation RUL MAE: 0.4991084933280945

>>> NEW BEST MAE — promoted to:
/Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp.keras


## 9. Test evaluation

Evaluate the final retained Phase 5 checkpoint once.

In [15]:
# ============================================================
# VALIDATION MAE HISTORY
# ============================================================
print("\nValidation RUL MAE by epoch:")
print("-" * 45)
for epoch, mae in enumerate(history.history["val_y_rul_hours_mae"], 1):
    print(f"Epoch {epoch:02d}: {mae:.6f} hours")

best_idx = int(np.argmin(history.history["val_y_rul_hours_mae"]))
print("-" * 45)
print(
    f"BEST = {history.history['val_y_rul_hours_mae'][best_idx]:.6f} hours "
    f"at epoch {best_idx + 1}"
)



Validation RUL MAE by epoch:
---------------------------------------------
Epoch 01: 0.761082 hours
Epoch 02: 0.647934 hours
Epoch 03: 0.757650 hours
Epoch 04: 0.556091 hours
Epoch 05: 0.544825 hours
Epoch 06: 0.514751 hours
Epoch 07: 0.499108 hours
Epoch 08: 0.513493 hours
Epoch 09: 0.714745 hours
Epoch 10: 0.702360 hours
Epoch 11: 0.756185 hours
Epoch 12: 0.843326 hours
---------------------------------------------
BEST = 0.499108 hours at epoch 7


In [16]:
if not os.path.exists(FINAL_CKPT):
    raise FileNotFoundError(
        f"Phase 5 checkpoint not found: {FINAL_CKPT}"
    )

test_ds = make_dataset(
    INDEX_PATH,
    DATA_DIR,
    "test",
    SCALER_PATH,
    batch_size=BATCH_SIZE,
    shuffle_buffer=0,
).map(to_multi_output_with_mask)

loaded_model = keras.models.load_model(
    '/Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp_temp.keras',
    safe_mode=False,
    custom_objects={
        "weighted_diagnosis_loss":
            weighted_diagnosis_loss,

        "diagnosis_accuracy":
            diagnosis_accuracy,

        "nonzero_severity_mse":
            nonzero_severity_mse,
    },
)

print("Evaluating:", FINAL_CKPT)

results = loaded_model.evaluate(
    test_ds,
    steps=TEST_STEPS,
    verbose=1,
    return_dict=True,
)

print()
print("========================================")
print("FINAL TEST RESULTS")
print("========================================")

for key, value in results.items():
    print(f"{key}: {value}")

Evaluating: /Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp.keras
522/522 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - loss: 0.5663 - y_detection_acc: 0.9837 - y_detection_loss: 0.0605 - y_diagnosis_diagnosis_accuracy: 0.9906 - y_diagnosis_loss: 3.7329e-04 - y_rul_hours_loss: 0.4652 - y_rul_hours_mae: 0.7370 - y_severity_loss: 0.0011 - y_severity_nonzero_severity_mse: 0.0054

FINAL TEST RESULTS
loss: 0.5662503242492676
y_detection_acc: 0.9836865663528442
y_detection_loss: 0.06048225238919258
y_diagnosis_diagnosis_accuracy: 0.9905654788017273
y_diagnosis_loss: 0.0003732923069037497
y_rul_hours_loss: 0.465248167514801
y_rul_hours_mae: 0.7370441555976868
y_severity_loss: 0.0010927696712315083
y_severity_nonzero_severity_mse: 0.005395513027906418


## Final architecture

```text
                     ┌── Frozen Phase-4 TCN ── Detection
                     │
Input (128 × 24) ────┼── Frozen Phase-4 TCN ── Diagnosis
                     │
                     ├── Frozen Phase-4 TCN ── Severity
                     │
                     └── Independent 2-layer LSTM ── RUL head
                                                        ↑
                                                  x_rul_aux (2)
```

Phase 4 remains untouched. Only the independent RUL LSTM branch and RUL head are trained in Phase 5.